# Imports

In [1]:
import sys
import torch
import os
import numpy as np
import nltk
sys.path.append('../..')
from nltk.corpus import stopwords
import pandas as pd
import plotly.express as px
from definitions import EXPERIMENTS_DIR
from typing import Optional, List, Dict, Tuple, Literal
from torch import Tensor
from src.commands.graph_learning.main import Trainer, Config, Graph
from src.commands.feature_extraction.graph_builder import EdgeTypes
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from torch_geometric import utils as gnn_utils
from torch_scatter import scatter
from torch_geometric.data import HeteroData
from functools import cache
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

In [2]:
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/fellah-a/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fellah-a/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
px.defaults.template = "seaborn"

In [4]:
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn

# Utils

In [140]:
def compute_entities_scores(
    results : Dict,
    graph : Graph,
    min_importance : float = 0.05
) -> Dict[str, Tensor]:

    E2A = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index
    r = {}

    for model, model_results in results.items():

        entities_weights = model_results['attn_weights'].flatten()

        entities_weights = gnn_utils.softmax(
            src=entities_weights[E2A[0].flatten()],
            index=E2A[1],
            dim=0,
            num_nodes=graph.pygeo['Article'].x.size(0)
        )

        entities_weights_mean = scatter(
            src=entities_weights,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='mean'
        )

        entities_weights_sum = scatter(
            src=entities_weights,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='sum'
        )

        degree = gnn_utils.degree(E2A[1], num_nodes=graph.pygeo['Article'].x.size(0))
        degree = 1 / degree
        degree = degree[E2A[1]]

        entities_weights_clipped_sum = scatter(
            src=entities_weights - min_importance,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='sum'
        )

        ### Relative Rank Score
        _, docs = E2A
        indices = torch.arange(len(docs))
        ranks = torch.zeros_like(E2A[0], dtype=torch.float)

        for doc in docs.unique():
            mask = docs == doc
            weights = entities_weights[mask]
            entities_indices = indices[mask]
            rank = weights.argsort(descending=True) + 1
            relative_rank = 1 - (rank.float() / mask.float().sum())
            ranks[entities_indices] = relative_rank

        print(ranks.size())

        entities_relative_rank = scatter(
            src=ranks,
            index=E2A[0],
            dim=0,
            dim_size=graph.pygeo['Entity'].x.size(0),
            reduce='mean'
        )

        r[model] = {
            "avg" : entities_weights_mean,
            "sum" : entities_weights_sum,
            "sum_clipped" : entities_weights_clipped_sum,
            "Sigmoid" : (model_results['attn_weights'] - model_results['attn_weights'].mean()).flatten().sigmoid(),
            'rank' : entities_relative_rank
        }

    return r

In [6]:
def compute_pseudo_labels(
    graph : Graph, 
    entities_weights : Tensor, 
    min_e_score : float,
    threshold : float | str,
    alpha : float = 2, # alpha >= 1
    freq_bounds : Tuple[float, float] = (0.01, 0.2)
) -> Tensor:

    X_a = graph.pygeo['Article'].x_passage
    X_e = graph.pygeo['Entity'].x
    entities_mask = entities_weights.flatten().gt(min_e_score)

    new_indices = torch.arange(entities_mask.sum(), device=entities_weights.device)
    indices_map = torch.full_like(entities_mask, -1, dtype=torch.long)
    indices_map[entities_mask] = new_indices

    entities, articles = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index

    articles = articles[entities_mask[entities]]
    entities = entities[entities_mask[entities]]
    entities = indices_map[entities]

    Y = X_a @ X_e[entities_mask,:].T

    if threshold == 'median':
        threshold = Y.median()
    elif threshold == 'mean':
        threshold = Y.mean()

    Y = (Y - Y.min()) / (Y.max() - Y.min() + 1e-8)
    threshold = (threshold - Y.min()) / (Y.max() - Y.min() + 1e-8)
    Y[articles, entities] = 1.0

    Y = torch.where(
        Y < threshold,
        Y ** alpha,
        1 - (1 - Y) ** alpha
    )

    freq = Y.gt(0.5).float().mean(dim=0)
    mask = (freq >= freq_bounds[0]) & (freq <= freq_bounds[1])

    Y = Y[:,mask]
    
    return Y

In [7]:
def get_label_stats(Y : Tensor, threshold : float = 0.5) -> Dict[str, float]:

    ths = [
        Y.quantile(0.25, dim=0).mean().item(), 
        Y.quantile(0.5, dim=0).mean().item(), 
        Y.quantile(0.75, dim=0).mean().item(), 
        Y.quantile(0.95, dim=0).mean().item()
    ]

    num_labels = Y.gt(threshold).sum(dim=1).float()

    quantiles = torch.quantile(
        num_labels,
        torch.tensor([0, .25, .5, .75, .95], device=Y.device)
    )

    return {
        "sparsity" : Y.gt(threshold).float().mean().item(),
        "Average Num. Docs / Label" : Y.gt(threshold).sum(dim=0).float().mean().item(),
        "Average Num. Labels / Doc" : Y.gt(threshold).sum(dim=1).float().mean().item(),
        "Mean" : Y.mean().item(),
        "Avg. Labels / Doc at Label-Value Quantiles [0,25; 0.5, 0.75, 0.95]" : [Y.gt(th).sum(dim=1).float().mean().item() for th in ths],
        "Quantiles [0,25; 0.5, 0.75, 0.95]" : ths,
        "Distribution of labels per document" : quantiles,
        "Num. Doc with zero labels" : Y.gt(threshold).sum(dim=-1).eq(0).sum().item(),
        "Num. Labels with zero doc (0)" : Y.gt(threshold).sum(dim=0).eq(0).sum().item(),
        "D" : torch.where(Y < 0.5, Y, 1 - Y).mean().item()
    }

In [8]:
def load_histories(path : str) -> pd.DataFrame:

    histories_dir = os.path.join(path, 'histories')
    histories = []

    for filename in os.listdir(histories_dir):
        filename = os.path.join(histories_dir, filename)
        df = pd.read_csv(filename)
        df['iteration'] = np.arange(len(df))
        histories.append(df)

    histories = pd.concat(histories)

    return histories #.groupby(by=['epoch', 'phase', 'config']).mean().reset_index()

In [9]:
def load_results(path : str) -> Dict[str, Dict[str, Tensor]]:

    results_dir = os.path.join(path, 'results')
    results = {}

    for filename in os.listdir(results_dir):
        result_path = os.path.join(results_dir, filename)
        exp = filename.split('.')[0].replace('predictions_', '')
        results[exp] = torch.load(result_path, map_location='cpu', weights_only=False)

    return results

In [10]:
@cache
def load_data(experiment : str) -> Tuple[Graph, HeteroData]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = trainer.load_data(config=config.training_config, return_raw=True)

    return graph, data

In [11]:
@cache
def load_model(experiment : str, config_name : str) -> Tuple[Graph, HeteroData, torch.nn.Module]:

    config = Config(experiment=experiment)
    trainer = Trainer(experiment=config.experiment, config=config.training_config, configs=['Attn'])

    data, graph = load_data(experiment=experiment)

    model_config = config.training_config.configs[config_name]
    model = trainer.create_model(config=model_config, data=graph)

    print(model.load_state_dict(torch.load(model_config.model_path, weights_only=False)))

    return model

In [12]:
def get_dataframes(graph : Graph) -> List[pd.DataFrame]:

    articles_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id,  **x.properties }, graph.get_nodes('Article').nodes_generator))).set_index('id')

    entities_df = pd.DataFrame(data=list(map(lambda x : { 'id' : x.id, **x.properties }, graph.get_nodes('Entity').nodes_generator)))
    
    triples_df = pd.DataFrame(data=list(map(lambda x : {
        "id" : x.id,
        "src" : x.src,
        "dst" : x.dst,
        "predicate" : x.properties['predicate']
    }, graph.get_edges(EdgeTypes.Entity_HASRELATIONWITH_Entity).edges_generator)))

    articles_df = articles_df \
        .join(pd.DataFrame(graph._edges[EdgeTypes.Article_HASKEYWORD_Keyword].edges_generator)[['src', 'dst']] \
        .groupby('src').agg(list)) \
        .rename(columns={'dst':'keywords'})

    return articles_df, entities_df, triples_df

In [13]:
def style_paper_plot(
    fig,
    *,
    width=950,
    height=600,
    font_family="Arial",
    font_size=14,
):
    fig.update_layout(
        width=width,
        height=height,
        font=dict(
            family=font_family,
            size=font_size,
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(
            l=80,
            r=30,
            t=30,
            b=70,
        ),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="lightgray",
        gridwidth=0.5,
        zeroline=False,
        ticks="outside",
        showline=True,
        linewidth=1,
        linecolor="black",
    )

    return fig

In [14]:
EXPERIMENT = "topic_modeling/pseudo_labeling"
EXPERIMENT_PATH = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
config = Config(experiment=EXPERIMENT)

In [15]:
history = load_histories(EXPERIMENT_PATH)
results = load_results(EXPERIMENT_PATH)

In [16]:
graph, data = load_data(experiment=EXPERIMENT)
# model = load_model(experiment=EXPERIMENT, config_name="baseline")

len(self.configs) = 1


/home/fellah-a/anaconda3/envs/text2kg-main/lib/python3.12/site-packages/torch_sparse/matmul.py:97: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  C = torch.sparse.mm(A, B)


In [17]:
articles_df, entities_df, triples_df = get_dataframes(graph)

In [141]:
entities_weights = compute_entities_scores(
    results=results,
    graph=graph,
    min_importance=0.05
)

torch.Size([93148])


In [24]:
Y = compute_pseudo_labels(
    graph=graph,
    entities_weights=entities_weights['baseline']['avg'],
    min_e_score=0.2,
    threshold=0.5,
    alpha=3,
    freq_bounds=(0.01, 0.2)
)
print(Y.shape)

torch.Size([7783, 3179])


In [19]:
get_label_stats(Y)

NameError: name 'Y' is not defined

In [142]:
# entities_weights['Attn']['Sigmoid_gated_clipped_sum'] = entities_weights['Attn']['Sigmoid'] * entities_weights['Attn']['sum_clipped']

for k, v in entities_weights['baseline'].items():
    entities_df[f'{k}_Score'] = v.numpy()
    entities_df[f'{k}_Rank'] = (-entities_df[f'{k}_Score']).rank().to_numpy()

In [117]:
subset_df = entities_df[entities_df['id'].isin([
    'knowledge_graph', 'deep_learning', 'graph_neural_network', 'internet_of_thing', 
    'model', 'data', 'france', 'nante', 'method', 'dataset', 'existing_cable_driven_parallel_robot',
    'quality', 'image_processing', 'video_quality', 'video_encoding', 'deepfake_detection_task',
    'state_of_the_art', 'state_of_the_art_method', 'sparql_query', 'food_waste', 'methodology',
])]
subset_df.drop(columns=['id', 'relevancy_scores', 'generality_scores', 'out_degree', 'in_degree', 'degree', 'pagerank', 'specificity_weighted_pagerank'])

,name,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
46,nante,0.091127,22650.0,0.637887,872.0,0.287887,2284.0,0.544634,31634.0,0.426189,38221.0
47,france,0.022837,54123.0,1.393059,95.0,-1.656940,58175.0,0.137586,56152.0,0.501474,31756.0
91,method,0.023436,54019.0,1.078065,189.0,-1.221935,58149.0,0.045772,57762.0,0.510121,31530.0
275,graph neural network,0.101489,18609.0,1.420839,91.0,0.720839,289.0,0.538634,33817.0,0.505166,31681.0
337,internet of thing,0.031465,52178.0,1.636179,61.0,-0.963821,58119.0,0.256022,54213.0,0.544592,29470.0
944,model,0.003855,57750.0,0.447122,2049.0,-5.352877,58198.0,0.032838,57987.0,0.518232,31225.0
2530,deep learning,0.049992,44994.0,2.449591,18.0,-0.000409,45177.0,0.369716,51769.0,0.473113,34832.0
2789,quality,0.000763,58195.0,0.010675,57312.0,-0.689325,58071.0,0.009309,58195.0,0.536116,30016.0
3347,dataset,0.005804,57353.0,0.185721,10633.0,-1.414279,58164.0,0.042768,57810.0,0.564817,27803.0
3423,state of the art,0.018291,54953.0,0.402405,2574.0,-0.697595,58074.0,0.169505,55629.0,0.529108,30589.0


In [88]:
entities_df[['degree', 'Sigmoid_Score']].corr()

,degree,Sigmoid_Score
degree,1.000000,-0.261635
Sigmoid_Score,-0.261635,1.000000


In [41]:
entities_df[entities_df.columns[entities_df.columns.str.endswith('_Rank')]].corr()

,avg_Rank,sum_Rank,sum_clipped_Rank,Sigmoid_Rank,rank_Rank
avg_Rank,1.000000,0.821230,0.962501,0.547791,0.007695
sum_Rank,0.821230,1.000000,0.885829,0.348224,0.018926
sum_clipped_Rank,0.962501,0.885829,1.000000,0.498180,0.009577
Sigmoid_Rank,0.547791,0.348224,0.498180,1.000000,-0.066182
rank_Rank,0.007695,0.018926,0.009577,-0.066182,1.000000


In [42]:
entities_df[entities_df.columns[entities_df.columns.str.endswith('_Score')]].corr()

,avg_Score,sum_Score,sum_clipped_Score,Sigmoid_Score,rank_Score
avg_Score,1.000000,0.572573,0.683323,0.401862,0.016193
sum_Score,0.572573,1.000000,0.704637,0.181903,0.019605
sum_clipped_Score,0.683323,0.704637,1.000000,0.438082,0.007424
Sigmoid_Score,0.401862,0.181903,0.438082,1.000000,-0.053189
rank_Score,0.016193,0.019605,0.007424,-0.053189,1.000000


In [43]:
def print_articles_entities_importance(
    article_id : int,
    articles_df : pd.DataFrame,
    entities_df : pd.DataFrame,
    score_col : str,
    graph : Graph
):

    article = articles_df.iloc[article_id]
    title = article['title']

    entities_ids, articles_ids = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index
    entities_ids = entities_ids[articles_ids == article_id].tolist()

    entities = entities_df.iloc[entities_ids]
    entities = pd.DataFrame.sort_values(entities, by=score_col, ascending=False)

    print(f"Title : {title} ")

    for idx, row in entities.iterrows():
        print(f"{row['name']} {{ 'score' : {row[score_col]} }}")

In [102]:
def print_nn_entities(
    entity_id : int,
    embeddings : Tensor,
    entities_df : pd.DataFrame,
    k : int = 5
):

    
    s = embeddings[[entity_id],:] @ embeddings.T
    s = s.argsort(descending=True).flatten().tolist()

    print('\n'.join(entities_df.iloc[s].head(k)['name'].tolist()))

In [112]:
entities_df[entities_df['name'].str.contains('cobot')]

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
18186,cobot,cobot,0.598265,0.494524,11.0,1.0,12.0,0.070258,0.134069,0.073071,31686.0,0.438426,2139.0,0.138426,7607.0,0.458478,47786.0,0.495289,32859.0
28048,mobile_cobot,mobile cobot,0.383986,0.000002,2.0,0.0,2.0,0.006689,0.040295,0.064756,36592.0,0.064756,40629.0,0.014756,37611.0,0.581533,14363.0,0.227273,48867.0


In [113]:
print_nn_entities(
    entity_id=18186,
    embeddings=results['baseline']['entities_embeddings'],
    entities_df=entities_df,
    k=5
)

optimization
cost
quality
energy
sufficiency


In [44]:
articles_df['idx'] = np.arange(len(articles_df))

In [45]:
articles_df.iloc[10]['abstract']

'Wi-Fi HaLow is expected to play a pivotal role in IoT 2.0; however, its increasing adoption makes it an attractive target for cyber adversaries. Enabling more than 8,000 devices on a single deployment, Wi-Fi HaLow awakens the possibility of massive network creation along 1 km distances. Unfortunately, attackers can easily take advantage of this extensive reach. Current research shows how an RTS attack, a MAC sub-layer attack, can shut down the communication exchange of an entire Wi-Fi HaLow network, yet no real test bed implementation has been done. This is the first article that studies the effects of an unsynchronized RTS attack on a hardware-implemented Wi-Fi HaLow network. The results show that the STA can not detect the attack based on the metrics and the attack has an impact of 95.7% at the physical layer and a 4.3% at the MAC sub-layer.'

In [74]:
print_articles_entities_importance(
    article_id=3398,
    articles_df=articles_df,
    entities_df=entities_df,
    score_col='avg_Score',
    graph=graph
)

Title : MILP model for fleet and charging infrastructure decisions for fast-charging city electric bus services 
state of the art method { 'score' : 0.1373184621334076 }
tested problem instance { 'score' : 0.10430970042943954 }
mixed integer linear programming model { 'score' : 0.09516292810440063 }
city of minsk { 'score' : 0.09232232719659805 }
fast charging electric bus { 'score' : 0.08483730256557465 }
efcient solution { 'score' : 0.0841660425066948 }
electrical bus { 'score' : 0.08283888548612595 }
transition to the sustainable city { 'score' : 0.07419770956039429 }
sustainable infrastructure { 'score' : 0.06819592416286469 }
sustainable public transportation { 'score' : 0.06793694198131561 }
greenhouse gas emission { 'score' : 0.06679921597242355 }
public bus transportation { 'score' : 0.06241857632994652 }
decision making process { 'score' : 0.05892803892493248 }
developed model { 'score' : 0.042381126433610916 }
framework { 'score' : 0.017541198059916496 }


In [47]:
fig = px.histogram(
    data_frame=entities_df,
    x='avg_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Average Document Attention Score")

fig.show()

In [48]:
fig = px.histogram(
    data_frame=entities_df,
    x='sum_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Sum Document Attention Score")

fig.show()

In [49]:
fig = px.histogram(
    data_frame=entities_df,
    x='sum_clipped_Score',
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Sum clipped Attention Score")

fig.show()

In [50]:
fig = px.histogram(
    data_frame=entities_df,
    x='Sigmoid_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Overall Attention Score")

fig.show()

In [143]:
fig = px.histogram(
    data_frame=entities_df,
    x='rank_Score'
)

fig = style_paper_plot(fig)
fig.update_layout(xaxis_title="Average Relative Rank")

fig.show()

In [52]:
def line_plot(entities_df : pd.DataFrame, col : str, n_steps : int = 20):

    data = []

    thresholds = np.arange(
        entities_df[col].min().item(), 
        entities_df[col].max().item(), 
        (entities_df[col].max().item() - entities_df[col].min().item()) / n_steps
    )

    for th in thresholds:

        N = (entities_df[col] >= th).astype(float).sum()
        p = 100 * (N / entities_df.shape[0])

        data.append({
            "threshold" : th,
            "%Entity" : p,
            'text' : f"{p:.2f}% ({N})"
        })

    data = pd.DataFrame(data)

    fig = px.line(
        data_frame=data,
        x="threshold",
        y="%Entity",
        markers=True,
    )

    fig = style_paper_plot(fig)
    fig.update_layout(xaxis_title=f"Threshold ({col.replace('_',' ')})", yaxis_title="%Entity (Size)")

    return fig

In [53]:
fig = line_plot(entities_df=entities_df, col='avg_Score', n_steps=50)
fig.show()

In [54]:
fig = line_plot(entities_df=entities_df, col='sum_Score', n_steps=50)
fig.show()

In [55]:
fig = line_plot(entities_df=entities_df, col='sum_clipped_Score', n_steps=200)
fig.show()

In [56]:
fig = line_plot(entities_df=entities_df, col='Sigmoid_Score', n_steps=50)
fig.show()

In [144]:
fig = line_plot(entities_df=entities_df, col='rank_Score', n_steps=50)
fig.show()

In [58]:
entities_df.sort_values(by='avg_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
35984,cesium_inserted_chromium_telluride,cesium inserted chromium telluride,0.240298,1.700144e-06,0.0,1.0,1.0,0.000951,0.034628,0.984882,1.0,0.984882,240.0,0.934882,90.0,0.574172,18038.0,0.5,31736.5
25561,thiazole_ring,thiazole ring,0.576707,1.700144e-06,0.0,1.0,1.0,0.002218,0.044291,0.984778,2.0,0.984778,241.0,0.934778,91.0,0.572044,19059.0,0.5,31736.5
55481,modeling_the_remediation_of_organic_contaminant,modeling the remediation of organic contaminant,0.344924,1.700144e-06,0.0,1.0,1.0,0.001510,0.038895,0.984704,3.0,0.984704,242.0,0.934704,92.0,0.549101,29819.0,0.5,31736.5
43340,thematic_map,thematic map,0.372602,1.133429e-06,1.0,0.0,1.0,0.000625,0.032145,0.980934,4.0,0.980934,245.0,0.930934,93.0,0.554257,27643.0,0.5,31736.5
13086,c_state,c state,0.599856,1.133429e-06,0.0,1.0,1.0,0.001165,0.036260,0.976244,5.0,0.976244,248.0,0.926244,97.0,0.552072,28636.0,0.5,31736.5
26350,set_of_admissible_solution,set of admissible solution,0.636825,1.700144e-06,0.0,1.0,1.0,0.001216,0.036653,0.971329,6.0,0.971329,252.0,0.921329,100.0,0.553769,27888.0,0.5,31736.5
42099,informal_transaction,informal transaction,0.736312,5.667147e-07,1.0,0.0,1.0,0.002366,0.045417,0.969015,7.0,0.969015,254.0,0.919015,101.0,0.572457,18831.0,0.5,31736.5
45141,multi_label_text_classification,multi label text classification,0.138066,1.700144e-06,1.0,0.0,1.0,0.001365,0.037784,0.968564,8.0,0.968564,255.0,0.918564,104.0,0.584232,13000.0,0.5,31736.5
3346,in_situ_datum,in situ datum,0.602068,1.133429e-06,1.0,0.0,1.0,0.001330,0.037518,0.966341,9.0,0.966341,257.0,0.916341,106.0,0.561923,23999.0,0.5,31736.5
56153,major_charge_carrier,major charge carrier,0.645595,1.700144e-06,1.0,0.0,1.0,0.001793,0.041050,0.962528,10.0,0.962528,261.0,0.912528,107.0,0.569811,20131.0,0.5,31736.5


In [35]:
entities_df.sort_values(by='sum_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
5566,existing_cable_driven_parallel_robot,existing cable driven parallel robot,0.294597,0.499862,87.0,20.0,107.0,0.558266,0.613138,0.115852,14095.0,5.676743,1.0,3.226743,1.0,0.547975,27720.0
3382,unit_cell,unit cell,0.590622,0.651918,36.0,16.0,52.0,0.333720,0.428578,0.143483,8835.0,4.304502,2.0,2.804502,2.0,0.534748,33078.0
4251,computation_time,computation time,0.624712,0.883298,5.0,24.0,29.0,0.164587,0.241597,0.143091,8891.0,3.434189,3.0,2.234190,4.0,0.550095,26784.0
2899,radiation_pattern,radiation pattern,0.540021,0.694962,10.0,18.0,28.0,0.147710,0.217787,0.145942,8510.0,3.356660,4.0,2.206660,5.0,0.544030,29404.0
314,full_wave_simulation,full wave simulation,0.510057,0.693319,19.0,17.0,36.0,0.196681,0.273103,0.114641,14416.0,3.324580,5.0,1.874580,8.0,0.514490,39188.0
7725,convolutional_neural_network,convolutional neural network,0.432221,0.741711,49.0,20.0,69.0,0.441082,0.532461,0.081691,26854.0,3.267639,6.0,1.267640,32.0,0.467788,46827.0
12158,transformation,transformation,0.685084,0.951048,9.0,7.0,16.0,0.089498,0.154918,0.323356,1152.0,3.233560,7.0,2.733560,3.0,0.528027,35347.0
4409,additive_manufacturing_technology,additive manufacturing technology,0.443513,0.785471,60.0,7.0,67.0,0.426938,0.518571,0.071977,32124.0,2.951038,8.0,0.901038,110.0,0.447782,48473.0
5881,reverberation_chamber,reverberation chamber,0.379326,0.592198,16.0,15.0,31.0,0.207623,0.302272,0.147119,8348.0,2.942390,9.0,1.942390,7.0,0.532636,33803.0
337,internet_of_thing,internet of thing,0.452033,0.769928,65.0,20.0,85.0,0.466401,0.536751,0.050267,44971.0,2.613868,10.0,0.013867,38073.0,0.360496,52482.0


In [36]:
entities_df.sort_values(by='sum_clipped_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
5566,existing_cable_driven_parallel_robot,existing cable driven parallel robot,0.294597,4.998617e-01,87.0,20.0,107.0,0.558266,0.613138,0.115852,14095.0,5.676743,1.0,3.226743,1.0,0.547975,27720.0
3382,unit_cell,unit cell,0.590622,6.519178e-01,36.0,16.0,52.0,0.333720,0.428578,0.143483,8835.0,4.304502,2.0,2.804502,2.0,0.534748,33078.0
12158,transformation,transformation,0.685084,9.510477e-01,9.0,7.0,16.0,0.089498,0.154918,0.323356,1152.0,3.233560,7.0,2.733560,3.0,0.528027,35347.0
4251,computation_time,computation time,0.624712,8.832980e-01,5.0,24.0,29.0,0.164587,0.241597,0.143091,8891.0,3.434189,3.0,2.234190,4.0,0.550095,26784.0
2899,radiation_pattern,radiation pattern,0.540021,6.949617e-01,10.0,18.0,28.0,0.147710,0.217787,0.145942,8510.0,3.356660,4.0,2.206660,5.0,0.544030,29404.0
7198,advanced_reflectarray_antenna_system,advanced reflectarray antenna system,0.254558,8.690146e-02,1.0,1.0,2.0,0.010194,0.057157,0.724656,109.0,2.173968,25.0,2.023968,6.0,0.578441,12660.0
5881,reverberation_chamber,reverberation chamber,0.379326,5.921977e-01,16.0,15.0,31.0,0.207623,0.302272,0.147119,8348.0,2.942390,9.0,1.942390,7.0,0.532636,33803.0
314,full_wave_simulation,full wave simulation,0.510057,6.933194e-01,19.0,17.0,36.0,0.196681,0.273103,0.114641,14416.0,3.324580,5.0,1.874580,8.0,0.514490,39188.0
13041,dynamic_regressor_extension_and_mixing_method,dynamic regressor extension and mixing method,0.290579,4.820051e-01,10.0,4.0,14.0,0.106319,0.196058,0.220164,3112.0,2.421809,16.0,1.871809,9.0,0.509460,40438.0
692,single_crack_propagation,single crack propagation,0.441246,6.559403e-01,8.0,16.0,24.0,0.149301,0.231585,0.152661,7728.0,2.595232,12.0,1.745232,10.0,0.607287,2905.0


In [37]:
entities_df.sort_values(by='Sigmoid_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
7531,chips_act,chips act,0.706055,2.316863e-01,1.0,2.0,3.0,0.012091,0.047379,0.076139,29763.0,0.456836,1904.0,0.156836,6243.0,0.706834,1.0
42209,strategies_developed_by_asean_region_airline,strategies developed by asean region airline,0.235820,1.983502e-06,0.0,8.0,8.0,0.052693,0.118406,0.619446,156.0,0.619446,936.0,0.569445,460.0,0.705356,2.0
44717,components_per_circuit,components per circuit,0.477142,1.133429e-06,1.0,0.0,1.0,0.001424,0.038237,0.683327,120.0,0.683327,728.0,0.633327,357.0,0.703875,3.0
34970,high_value_added_application,high value added application,0.697468,1.983502e-06,0.0,1.0,1.0,0.002780,0.048574,0.566209,206.0,0.566209,1163.0,0.516209,596.0,0.703199,4.0
28083,refrigerated_warehousing_of_food,refrigerated warehousing of food,0.433797,1.700144e-06,0.0,1.0,1.0,0.001965,0.042363,0.178201,5289.0,0.178201,11408.0,0.128201,8344.0,0.701495,5.0
53981,marc_caro,marc caro,0.457428,1.700144e-06,1.0,0.0,1.0,0.002069,0.043159,0.322642,1163.0,0.322642,4001.0,0.272642,2460.0,0.701033,6.0
54493,minimum_bounding_rectangle,minimum bounding rectangle,0.346815,1.700144e-06,1.0,0.0,1.0,0.002119,0.043535,0.339247,1012.0,0.339247,3619.0,0.289247,2215.0,0.700620,7.0
37140,local_transport_policy,local transport policy,0.504953,1.700144e-06,0.0,1.0,1.0,0.001562,0.039293,0.098875,19215.0,0.098875,26127.0,0.048875,22408.0,0.696884,8.0
8684,modifications_of_security_and_reliability,modifications of security and reliability,0.420805,4.379062e-02,0.0,2.0,2.0,0.008654,0.049751,0.245727,2360.0,0.491453,1615.0,0.391453,1165.0,0.695228,9.0
23865,transition_period,transition period,0.570672,1.133429e-06,1.0,0.0,1.0,0.005419,0.068699,0.377516,758.0,0.377516,2891.0,0.327516,1726.0,0.694304,10.0


In [145]:
entities_df.sort_values(by='rank_Score', ascending=False).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
18538,optical_guide,optical guide,0.472356,0.000001,1.0,1.0,2.0,0.009969,0.056076,0.012121,56097.0,0.012121,57158.0,-0.037879,54390.0,0.471807,46699.0,0.987179,1.0
36677,versatile_mobile_robot,versatile mobile robot,0.437748,0.000002,0.0,1.0,1.0,0.005238,0.067324,0.020898,54491.0,0.020898,55942.0,-0.029102,53203.0,0.593058,8999.0,0.986486,2.0
51326,flowing_property,flowing property,0.635708,0.000002,0.0,1.0,1.0,0.002279,0.044757,0.014291,55720.0,0.014291,56876.0,-0.035709,54140.0,0.497371,43683.0,0.985915,3.0
33436,dynamics_model_constraint,dynamics model constraint,0.577768,0.000001,1.0,0.0,1.0,0.003727,0.055796,0.015209,55537.0,0.015209,56758.0,-0.034791,54025.0,0.466478,47142.0,0.984615,4.0
17411,on_resistance,on resistance,0.740778,0.000002,0.0,1.0,1.0,0.002110,0.043472,0.006141,57277.0,0.006141,57822.0,-0.043859,55041.0,0.260314,54147.0,0.984127,5.0
8143,ndsi,ndsi,0.510305,0.000001,1.0,1.0,2.0,0.008096,0.047066,0.023744,53963.0,0.023744,55509.0,-0.026256,52802.0,0.584393,12920.0,0.982143,6.0
16354,coordinating_complementary_currency_community,coordinating complementary currency community,0.362732,0.000002,0.0,1.0,1.0,0.001906,0.041911,0.032993,51759.0,0.032993,53628.0,-0.017007,50925.0,0.610052,3516.5,0.981481,7.0
19777,mixed_ionic_electronic_conductor,mixed ionic electronic conductor,0.299821,0.000002,0.0,1.0,1.0,0.002852,0.049129,0.030398,52479.0,0.030398,54230.0,-0.019602,51527.0,0.586317,12019.0,0.978261,8.0
23389,strategic_shift,strategic shift,0.638129,0.000002,1.0,1.0,2.0,0.006050,0.037220,0.039534,49541.0,0.039534,51727.0,-0.010466,49009.0,0.607966,4008.0,0.977273,9.5
41110,thermochromic_property,thermochromic property,0.277840,0.000002,1.0,0.0,1.0,0.001753,0.040744,0.027241,53233.0,0.027241,54880.0,-0.022759,52184.0,0.525154,37926.0,0.977273,9.5


In [135]:
e,a = graph.pygeo['Entity', 'APPEARIN', 'Article'].edge_index

In [136]:
a[e == 944].unique()

tensor([  82,   87,  112,  128,  235,  326,  334,  352,  450,  568,  582,  597,
         609,  641,  750,  759,  960, 1070, 1106, 1239, 1298, 1500, 1537, 1603,
        1740, 1748, 1890, 1980, 2051, 2055, 2134, 2204, 2319, 2349, 2498, 2511,
        2533, 2539, 2586, 2619, 2620, 2686, 2855, 2938, 3011, 3046, 3307, 3407,
        3530, 3545, 3567, 3676, 3822, 3915, 3931, 3963, 4037, 4226, 4229, 4235,
        4269, 4325, 4350, 4373, 4433, 4449, 4502, 4612, 4671, 4852, 4873, 4894,
        4910, 4912, 4931, 4935, 5101, 5111, 5323, 5357, 5375, 5404, 5462, 5470,
        5644, 5680, 5751, 5867, 5911, 5989, 6012, 6027, 6070, 6145, 6185, 6260,
        6342, 6344, 6368, 6418, 6443, 6738, 6789, 6882, 6970, 6993, 7022, 7159,
        7243, 7356, 7431, 7500, 7545, 7602, 7666, 7753])

In [134]:
a[e == 18186].unique()

tensor([], dtype=torch.int64)

In [139]:
print_articles_entities_importance(
    article_id=112,
    articles_df=articles_df,
    entities_df=entities_df,
    score_col='avg_Score',
    graph=graph
)

Title : Microwave-assisted desorption of CO<sub>2</sub> from ethanolamine solvent in a porous hollow fiber membrane - numerical modeling 
single fiber system { 'score' : 0.2671813666820526 }
microwave solvent regeneration { 'score' : 0.2351612001657486 }
isothermal regeneration { 'score' : 0.21007025241851807 }
microwave mode { 'score' : 0.19749027490615845 }
hotspot { 'score' : 0.07154032588005066 }
fiber { 'score' : 0.005322783719748259 }
model { 'score' : 0.003854503622278571 }


In [148]:
entities_df.sort_values(by='Sigmoid_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
2200,optimization,optimization,0.625322,9.268087e-01,18.0,24.0,42.0,0.233373,0.312764,0.000685,58197.0,0.022595,55687.0,-1.627405,58173.0,0.007094,58198.0,0.432813,31495.0
704,cost,cost,0.725992,9.210024e-01,8.0,34.0,42.0,0.218821,0.292313,0.000766,58194.0,0.025264,55275.0,-1.624736,58172.0,0.007719,58197.0,0.410369,33257.0
2434,analysis,analysis,0.685900,9.275593e-01,8.0,18.0,26.0,0.139064,0.209180,0.000939,58185.0,0.020662,55977.0,-1.079338,58133.0,0.009141,58196.0,0.365126,36482.0
2789,quality,quality,0.751090,9.212569e-01,5.0,11.0,16.0,0.078610,0.134606,0.000763,58195.0,0.010675,57312.0,-0.689325,58071.0,0.009309,58195.0,0.470668,28556.0
2898,measurement,measurement,0.711675,8.740606e-01,25.0,32.0,57.0,0.271066,0.336951,0.001235,58172.0,0.054347,45828.0,-2.145653,58187.0,0.009544,58194.0,0.343897,37907.0
1040,energy,energy,0.732109,9.177368e-01,11.0,19.0,30.0,0.148313,0.214004,0.001373,58154.0,0.037083,52510.0,-1.312917,58155.0,0.009573,58193.0,0.371912,36150.0
488,sufficiency,sufficiency,0.574905,1.663421e-02,1.0,1.0,2.0,0.006459,0.039190,0.000886,58188.0,0.001772,58168.0,-0.098228,56623.0,0.009968,58192.0,0.644886,16532.0
27708,price,price,0.713588,7.526116e-01,2.0,4.0,6.0,0.030002,0.075266,0.000785,58192.0,0.004711,57966.0,-0.295289,57728.0,0.010262,58191.0,0.382244,35380.0
20316,functionality,functionality,0.684105,8.240710e-01,0.0,6.0,6.0,0.028296,0.070632,0.000725,58196.0,0.003626,58053.0,-0.246374,57601.0,0.010268,58190.0,0.336432,38107.0
12855,calculation,calculation,0.761406,8.668795e-01,5.0,5.0,10.0,0.057929,0.117961,0.000928,58187.0,0.004642,57973.0,-0.245358,57597.0,0.010759,58189.0,0.723137,11980.0


In [146]:
entities_df.sort_values(by='rank_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank,rank_Score,rank_Rank
48277,randomly_defined_population,randomly defined population,0.766864,1.700144e-06,1.0,0.0,1.0,0.002088,0.043298,0.129941,11308.0,0.129941,18327.0,0.079941,14869.0,0.559770,25048.0,0.0,56501.0
48325,ris_empowered_wireless_communication,ris empowered wireless communication,0.241054,1.700144e-06,2.0,0.0,2.0,0.013752,0.074277,0.062302,38061.0,0.062302,41884.0,0.012302,38903.0,0.577139,16560.0,0.0,56501.0
2640,decentralized_pi_controller,decentralized pi controller,0.480917,1.700144e-06,1.0,0.0,1.0,0.000770,0.033249,0.095769,20734.0,0.095769,27134.0,0.045769,23726.0,0.556836,26459.0,0.0,56501.0
48339,accessibility_concern,accessibility concern,0.800505,1.133429e-06,0.0,1.0,1.0,0.001439,0.038352,0.183904,4968.0,0.183904,10801.0,0.133904,7988.0,0.599505,6559.0,0.0,56501.0
48356,nodal_model,nodal model,0.636089,1.700144e-06,0.0,1.0,1.0,0.003754,0.056007,0.454110,447.0,0.454110,1979.0,0.404110,1149.0,0.577768,16235.0,0.0,56501.0
48381,toxicology,toxicology,0.568190,1.700144e-06,0.0,1.0,1.0,0.001956,0.042293,0.020089,54639.0,0.020089,56047.0,-0.029911,53317.0,0.265327,54054.0,0.0,56501.0
48374,clay_brick,clay brick,0.564590,1.700144e-06,2.0,0.0,2.0,0.007520,0.044293,0.132212,10895.0,0.132212,17879.0,0.082212,14443.0,0.562692,23633.0,0.0,56501.0
35771,guest_adsorption_desorption_property,guest adsorption desorption property,0.520592,5.667147e-07,0.0,1.0,1.0,0.002425,0.045870,0.071686,32514.0,0.071686,37170.0,0.021686,34062.0,0.568895,20564.0,0.0,56501.0
35765,array_of_septum_polarizer,array of septum polarizer,0.301973,1.700144e-06,1.0,0.0,1.0,0.001527,0.039025,0.214860,3400.0,0.214860,8461.0,0.164860,5964.0,0.557024,26365.0,0.0,56501.0
36000,mechanical_locking,mechanical locking,0.764664,1.700144e-06,1.0,1.0,2.0,0.006489,0.039334,0.080995,27499.0,0.080995,32904.0,0.030995,29708.0,0.604148,5092.0,0.0,56501.0


In [180]:
entities_df.sort_values(by='avg_Score', ascending=True).head(20)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
25933,data_transfer,data transfer,0.617191,6.913016e-01,6.0,5.0,11.0,0.069337,0.137227,0.000003,58198.0,0.000023,58167.0,-0.349977,57982.0,0.000048,58194.0
17975,modification,modification,0.659987,7.104951e-01,2.0,2.0,4.0,0.022572,0.071790,0.000004,58197.0,0.000011,58187.0,-0.149989,57734.0,0.000046,58196.0
63,temperature,temperature,0.666039,9.179422e-01,48.0,68.0,116.0,0.603907,0.653073,0.000004,58196.0,0.000327,57624.0,-4.549673,58196.0,0.000047,58195.0
8162,orbit,orbit,0.870521,1.133429e-06,0.0,1.0,1.0,0.002786,0.048622,0.000004,58195.0,0.000004,58198.0,-0.049996,57290.0,0.000233,58149.0
2434,analysis,analysis,0.685900,9.275593e-01,8.0,18.0,26.0,0.139064,0.209180,0.000004,58194.0,0.000086,58025.0,-1.099914,58151.0,0.000042,58197.0
48658,dreaming,dreaming,0.536889,1.700144e-06,0.0,1.0,1.0,0.002767,0.048479,0.000005,58193.0,0.000005,58197.0,-0.049995,57289.0,0.000035,58198.0
5574,data_transmission,data transmission,0.604290,7.595649e-01,1.0,9.0,10.0,0.052813,0.106683,0.000005,58192.0,0.000046,58116.0,-0.449953,58049.0,0.000110,58189.0
41163,tool_wear,tool wear,0.642708,5.667147e-07,1.0,0.0,1.0,0.003162,0.051490,0.000006,58191.0,0.000006,58196.0,-0.049994,57288.0,0.000117,58187.0
45554,contract,contract,0.684798,1.133429e-06,0.0,1.0,1.0,0.003511,0.054151,0.000007,58190.0,0.000007,58195.0,-0.049993,57287.0,0.000098,58193.0
11197,heat,heat,0.721628,7.462979e-01,0.0,11.0,11.0,0.054769,0.106239,0.000007,58189.0,0.000060,58073.0,-0.449940,58048.0,0.000108,58192.0


In [39]:
entities_df[entities_df['name'].str.contains('query')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
23443,query_selection_algorithms,query selection algorithms,0.444244,1.700144e-06,0.0,1.0,1.0,0.001448,0.038416,0.322515,1168.0,0.322515,4006.0,0.272515,2468.0,0.550305,26688.0
6629,hybrid_query_relaxation_solution,hybrid query relaxation solution,0.174377,1.700144e-06,2.0,0.0,2.0,0.010608,0.059152,0.278834,1694.0,0.278834,5277.0,0.228834,3366.0,0.593826,6358.0
16229,count_distinct_sparql_query,count distinct sparql query,0.068806,5.667147e-07,2.0,2.0,4.0,0.016451,0.051689,0.208605,3581.0,0.208605,8817.0,0.158605,6141.0,0.568707,17638.0
3814,continuation_query,continuation query,0.561172,1.983502e-06,0.0,1.0,1.0,0.003581,0.054688,0.191179,4484.0,0.191179,10198.0,0.141179,7331.0,0.565821,19041.0
37625,complex_model_query,complex model query,0.578375,1.700144e-06,1.0,0.0,1.0,0.003512,0.054162,0.189372,4574.0,0.189372,10335.0,0.139372,7449.0,0.576221,13793.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27780,query_processing,query processing,0.537663,3.752320e-01,0.0,2.0,2.0,0.012592,0.068696,0.031996,52437.0,0.063992,41084.0,-0.036008,54550.0,0.327003,53361.0
27248,exploratory_query,exploratory query,0.608194,1.983502e-06,0.0,1.0,1.0,0.002426,0.045874,0.031945,52458.0,0.031945,54276.0,-0.018055,51549.0,0.319251,53516.0
14114,query_language,query language,0.443695,1.133429e-06,0.0,1.0,1.0,0.003127,0.051226,0.031472,52610.0,0.031472,54405.0,-0.018528,51681.0,0.264718,54612.0
6584,query_image,query image,0.477579,5.667147e-07,0.0,1.0,1.0,0.001741,0.040655,0.029274,53184.0,0.029274,54865.0,-0.020726,52152.0,0.464447,47161.0


In [40]:
entities_df[entities_df['name'].str.contains('query')].sort_values(by='sum_clipped_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
23443,query_selection_algorithms,query selection algorithms,0.444244,1.700144e-06,0.0,1.0,1.0,0.001448,0.038416,0.322515,1168.0,0.322515,4006.0,0.272515,2468.0,0.550305,26688.0
6629,hybrid_query_relaxation_solution,hybrid query relaxation solution,0.174377,1.700144e-06,2.0,0.0,2.0,0.010608,0.059152,0.278834,1694.0,0.278834,5277.0,0.228834,3366.0,0.593826,6358.0
3812,sparql_query_engine,sparql query engine,0.363612,3.373602e-01,0.0,2.0,2.0,0.010079,0.056607,0.163778,6543.0,0.327556,3891.0,0.227556,3398.0,0.570109,16919.0
9178,sparql_query,sparql query,0.414664,5.587765e-01,3.0,9.0,12.0,0.073349,0.140439,0.072675,31695.0,0.654075,818.0,0.204075,4096.0,0.475916,45921.0
31869,complex_sparql_property_path_query,complex sparql property path query,0.156291,2.647714e-01,5.0,3.0,8.0,0.047092,0.104931,0.099159,19095.0,0.396636,2613.0,0.196636,4359.0,0.553559,25184.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27248,exploratory_query,exploratory query,0.608194,1.983502e-06,0.0,1.0,1.0,0.002426,0.045874,0.031945,52458.0,0.031945,54276.0,-0.018055,51549.0,0.319251,53516.0
14114,query_language,query language,0.443695,1.133429e-06,0.0,1.0,1.0,0.003127,0.051226,0.031472,52610.0,0.031472,54405.0,-0.018528,51681.0,0.264718,54612.0
6584,query_image,query image,0.477579,5.667147e-07,0.0,1.0,1.0,0.001741,0.040655,0.029274,53184.0,0.029274,54865.0,-0.020726,52152.0,0.464447,47161.0
27780,query_processing,query processing,0.537663,3.752320e-01,0.0,2.0,2.0,0.012592,0.068696,0.031996,52437.0,0.063992,41084.0,-0.036008,54550.0,0.327003,53361.0


In [41]:
entities_df[entities_df['name'].str.contains('cloud')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
32247,edge_and_multi_cloud_environment,edge and multi cloud environment,0.357042,1.700144e-06,0.0,1.0,1.0,0.004416,0.061056,0.276657,1730.0,0.276657,5355.0,0.226657,3422.0,0.604956,3343.0
19030,point_cloud_quality_assessment,point cloud quality assessment,0.329363,2.266859e-06,1.0,1.0,2.0,0.008340,0.048240,0.255550,2126.0,0.255550,6229.0,0.205550,4060.0,0.575536,14161.0
8581,cloud_platform,cloud platform,0.492914,2.867579e-01,0.0,2.0,2.0,0.011941,0.065566,0.212903,3403.0,0.425806,2250.0,0.325806,1753.0,0.564166,19900.0
41952,cloud_related_architecture,cloud related architecture,0.264816,1.133429e-06,2.0,3.0,5.0,0.016748,0.044779,0.200078,4005.0,0.200078,9467.0,0.150078,6696.0,0.581375,11295.0
2989,cloud_infrastructure_layer,cloud infrastructure layer,0.409661,1.700144e-06,0.0,1.0,1.0,0.001500,0.038814,0.197535,4123.0,0.197535,9662.0,0.147535,6852.0,0.591784,7056.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27662,cloud_server,cloud server,0.608422,2.565903e-01,4.0,1.0,5.0,0.025103,0.069425,0.041212,49256.0,0.082425,32347.0,-0.017575,51435.0,0.509715,40383.0
51826,layered_cloud_networked_manufacturing_perspective,layered cloud networked manufacturing perspective,0.378953,5.667147e-07,2.0,1.0,3.0,0.018262,0.070910,0.038464,50333.0,0.038464,52505.0,-0.011536,49738.0,0.486332,44582.0
47296,cloud_resource_allocation,cloud resource allocation,0.663933,2.266859e-06,1.0,0.0,1.0,0.003720,0.055745,0.023392,54481.0,0.023392,55905.0,-0.026608,53211.0,0.458559,47642.0
8135,cloud_cover,cloud cover,0.672245,1.133429e-06,1.0,1.0,2.0,0.014061,0.075766,0.015128,56089.0,0.015128,57134.0,-0.034872,54435.0,0.468170,46784.0


In [42]:
entities_df[entities_df['name'].str.contains('federated')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
13117,statistical_concerns_in_federated_learning,statistical concerns in federated learning,0.218752,0.000002,0.0,1.0,1.0,0.001028,0.035216,0.108259,16157.0,0.108259,23329.0,0.058259,19557.0,0.592841,6695.0
45459,federated_query,federated query,0.362544,0.358976,1.0,3.0,4.0,0.022337,0.071020,0.101917,18123.0,0.305750,4445.0,0.155750,6306.0,0.532985,33686.0
20675,fully_decentralized_hierarchical_federated_lea...,fully decentralized hierarchical federated lea...,0.275796,0.000002,0.0,1.0,1.0,0.003150,0.051398,0.099123,19115.0,0.099123,26040.0,0.049123,22321.0,0.601752,4100.0
20697,classical_federated_learning_model,classical federated learning model,0.377624,0.000002,0.0,1.0,1.0,0.001599,0.039569,0.096321,20182.0,0.096321,26990.0,0.046321,23297.0,0.597766,5136.0
21948,federated_online_shopping_application,federated online shopping application,0.237490,0.000002,0.0,1.0,1.0,0.004742,0.063538,0.095967,20302.0,0.095967,27093.0,0.045967,23403.0,0.595114,5936.0
13115,federated_learning_framework,federated learning framework,0.289614,0.000002,1.0,1.0,2.0,0.006243,0.038148,0.095894,20333.0,0.095894,27117.0,0.045894,23428.0,0.563268,20340.0
21951,virtual_federated_store,virtual federated store,0.363551,0.000002,1.0,1.0,2.0,0.010979,0.060936,0.082981,26195.0,0.082981,32101.0,0.032981,28655.0,0.559652,22220.5
33054,federated_learning_with_aggregator,federated learning with aggregator,0.126546,0.000002,0.0,1.0,1.0,0.001362,0.037762,0.081262,27062.0,0.081262,32858.0,0.031262,29437.0,0.582066,11005.0
33051,standard_federated_learning_scalability,standard federated learning scalability,0.153681,0.000002,0.0,1.0,1.0,0.001362,0.037762,0.079915,27783.0,0.079915,33454.0,0.029915,30046.0,0.577995,12886.0
33055,federated_learning_with_election,federated learning with election,0.129985,0.000002,0.0,1.0,1.0,0.001362,0.037762,0.079838,27819.0,0.079838,33487.0,0.029838,30079.0,0.577761,13016.0


In [43]:
entities_df[entities_df['name'].str.contains('recommender')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
29443,recommender_system,recommender system,0.603064,0.576216,3.0,1.0,4.0,0.021539,0.068399,0.267145,1896.0,0.801435,469.0,0.651435,338.0,0.589816,7760.0


In [44]:
entities_df[entities_df['name'].str.contains('trajectory')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
2667,sets_of_trajectory,sets of trajectory,0.448852,0.424620,0.0,2.0,2.0,0.009081,0.051803,0.371230,790.0,0.742459,581.0,0.642459,350.0,0.570645,16627.0
49196,different_points_of_the_trajectory,different points of the trajectory,0.658858,0.000002,0.0,1.0,1.0,0.001964,0.042354,0.287828,1569.0,0.287828,4984.0,0.237828,3157.0,0.621604,1083.0
27184,motion_trajectory,motion trajectory,0.653903,0.000002,1.0,1.0,2.0,0.009293,0.052824,0.229457,2787.0,0.229457,7493.0,0.179457,5019.0,0.541853,30322.0
21227,baseline_trajectory,baseline trajectory,0.482230,0.000002,1.0,0.0,1.0,0.001055,0.035426,0.204464,3784.0,0.204464,9126.0,0.154464,6395.0,0.532250,33955.0
55130,trajectory_planning_method,trajectory planning method,0.422160,0.000002,3.0,0.0,3.0,0.023058,0.089195,0.198661,4066.0,0.198661,9575.0,0.148661,6786.0,0.558955,22569.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9192,dynamic_trajectory,dynamic trajectory,0.462277,0.000002,0.0,2.0,2.0,0.010398,0.058141,0.049471,45366.0,0.049471,48237.0,-0.000529,45339.0,0.504056,41586.0
9194,trajectory_optimization_method,trajectory optimization method,0.364454,0.000002,1.0,0.0,1.0,0.002322,0.045084,0.044021,48042.0,0.044021,50545.0,-0.005979,47710.0,0.474895,46041.0
46912,linear_trajectory,linear trajectory,0.690089,0.000002,1.0,1.0,2.0,0.010238,0.057371,0.040219,49673.0,0.040219,51949.0,-0.009781,49162.0,0.463800,47226.0
15890,trajectory,trajectory,0.626559,0.837438,2.0,10.0,12.0,0.079707,0.153540,0.028077,53461.0,0.280773,5209.0,-0.219227,57572.0,0.241695,55013.0


In [45]:
entities_df[entities_df['name'].str.contains('sparql')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
3792,sparql_federation_engine,sparql federation engine,0.230511,2.372682e-01,2.0,2.0,4.0,0.026964,0.086214,0.231782,2710.0,0.463565,1856.0,0.363565,1391.0,0.501007,42219.0
16229,count_distinct_sparql_query,count distinct sparql query,0.068806,5.667147e-07,2.0,2.0,4.0,0.016451,0.051689,0.208605,3581.0,0.208605,8817.0,0.158605,6141.0,0.568707,17638.0
3812,sparql_query_engine,sparql query engine,0.363612,3.373602e-01,0.0,2.0,2.0,0.010079,0.056607,0.163778,6543.0,0.327556,3891.0,0.227556,3398.0,0.570109,16919.0
10515,sparql_continuation_query,sparql continuation query,0.266895,1.983502e-06,0.0,1.0,1.0,0.002570,0.046977,0.160996,6819.0,0.160996,13368.0,0.110996,10097.0,0.579210,12293.0
21940,sparql_standard,sparql standard,0.466581,1.133429e-06,1.0,0.0,1.0,0.002422,0.045851,0.153582,7601.0,0.153582,14327.0,0.103582,10948.0,0.522732,36969.0
10513,sparql_endpoint_provider,sparql endpoint provider,0.380041,1.983502e-06,1.0,0.0,1.0,0.003868,0.056877,0.142718,8947.0,0.142718,15969.0,0.092718,12448.0,0.549592,27008.0
52300,regular_sparql_query,regular sparql query,0.385354,1.700144e-06,0.0,1.0,1.0,0.002634,0.047464,0.139681,9414.0,0.139681,16497.0,0.089681,12925.0,0.568993,17479.0
31872,traditional_sparql_query,traditional sparql query,0.336221,1.700144e-06,0.0,2.0,2.0,0.014428,0.077531,0.123174,12385.0,0.123174,19670.0,0.073174,15952.0,0.522328,37115.0
54010,decentralized_sparql_query_processing,decentralized sparql query processing,0.227607,1.133429e-06,2.0,0.0,2.0,0.007374,0.043592,0.122881,12443.0,0.122881,19725.0,0.072881,16008.0,0.583275,10464.0
3808,online_public_sparql_endpoint,online public sparql endpoint,0.352153,5.221259e-01,2.0,5.0,7.0,0.043115,0.102172,0.118845,13355.0,0.713068,648.0,0.413068,1030.0,0.498382,42711.0


In [46]:
entities_df[entities_df['name'].str.contains('large')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
57474,large_scale_finite_element_analysis,large scale finite element analysis,0.445491,1.700144e-06,0.0,1.0,1.0,0.003017,0.050384,0.599107,164.0,0.599107,1000.0,0.549107,497.0,0.609568,2501.0
50585,large_number_of_point,large number of point,0.496099,2.266859e-06,0.0,1.0,1.0,0.001255,0.036950,0.568519,202.0,0.568519,1148.0,0.518519,590.0,0.556360,23879.0
14879,large_amounts_of_training_datum,large amounts of training datum,0.524993,1.700144e-06,0.0,1.0,1.0,0.002161,0.043856,0.431877,502.0,0.431877,2185.0,0.381877,1229.0,0.533883,33369.0
50213,silicon_nanowire_with_larger_diameter,silicon nanowire with larger diameter,0.271955,5.667147e-07,2.0,0.0,2.0,0.011131,0.061665,0.340675,990.0,0.340675,3580.0,0.290675,2180.0,0.606693,3001.0
17612,large_and_positive_effect,large and positive effect,0.509450,1.700144e-06,0.0,3.0,3.0,0.020821,0.080665,0.335632,1034.0,0.335632,3700.0,0.285632,2262.0,0.553577,25176.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20549,large_quantity,large quantity,0.678007,5.543026e-01,0.0,2.0,2.0,0.008897,0.050920,0.016436,55859.0,0.032873,54057.0,-0.067127,56089.0,0.189255,55831.0
20864,large_scale_investment,large scale investment,0.798433,1.363544e-02,4.0,0.0,4.0,0.028116,0.089999,0.016374,55871.0,0.032749,54078.0,-0.067251,56092.0,0.428748,49746.0
39742,large_part,large part,0.666105,3.195648e-01,0.0,2.0,2.0,0.007298,0.043225,0.012742,56459.0,0.025483,55570.0,-0.074517,56261.0,0.129286,56700.0
6093,large_scale,large scale,0.697055,6.823229e-01,0.0,3.0,3.0,0.014555,0.056774,0.012709,56464.0,0.038126,52608.0,-0.111874,56882.0,0.161905,56240.0


In [47]:
entities_df[entities_df['name'].str.contains('model')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
55481,modeling_the_remediation_of_organic_contaminant,modeling the remediation of organic contaminant,0.344924,0.000002,0.0,1.0,1.0,0.001510,0.038895,0.977222,3.0,0.977222,239.0,0.927222,91.0,0.552540,25669.0
31558,pre_trained_language_model_camembert,pre trained language model camembert,0.382875,0.000002,0.0,1.0,1.0,0.001627,0.039783,0.907953,34.0,0.907953,321.0,0.857953,145.0,0.562583,20691.0
57481,moderate_to_severe_plant_model_mismatch,moderate to severe plant model mismatch,0.483221,0.000001,0.0,1.0,1.0,0.000771,0.033255,0.841276,62.0,0.841276,407.0,0.791276,196.0,0.608935,2600.0
25381,designed_filter_model,designed filter model,0.440269,0.000001,0.0,1.0,1.0,0.004141,0.058955,0.778654,88.0,0.778654,513.0,0.728654,261.0,0.581826,11109.0
10967,hierarchical_bayesian_model,hierarchical bayesian model,0.318415,0.000001,1.0,0.0,1.0,0.003680,0.055438,0.732585,104.0,0.732585,601.0,0.682585,305.0,0.596379,5532.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21191,modelling,modelling,0.630673,0.827155,8.0,4.0,12.0,0.061560,0.116145,0.005173,57711.0,0.051731,47239.0,-0.448269,57970.0,0.063460,57660.0
13106,thermodynamic_modeling,thermodynamic modeling,0.560697,0.408693,2.0,0.0,2.0,0.010115,0.056778,0.004863,57751.0,0.009726,57633.0,-0.090274,56599.0,0.056282,57755.0
54265,different_model,different model,0.771613,0.000002,2.0,0.0,2.0,0.010020,0.056321,0.004407,57827.0,0.004407,58072.0,-0.045593,55470.0,0.076906,57485.0
3284,modeling,modeling,0.673008,0.914513,12.0,7.0,19.0,0.093389,0.152033,0.004025,57878.0,0.060372,43026.0,-0.689628,58085.0,0.035594,58022.0


In [48]:
entities_df[entities_df['name'].str.contains('method')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
13754,precise_methodological_tool,precise methodological tool,0.426355,1.133429e-06,1.0,0.0,1.0,0.002761,0.048436,0.911548,32.0,0.911548,314.0,0.861548,140.0,0.545623,28743.0
49052,single_scan_method,single scan method,0.303337,2.266859e-06,1.0,0.0,1.0,0.003009,0.050321,0.664983,130.0,0.664983,790.0,0.614983,385.0,0.585209,9621.0
35090,explanation_method,explanation method,0.442141,3.295747e-01,0.0,3.0,3.0,0.013354,0.052196,0.447983,450.0,0.895966,337.0,0.795966,193.0,0.604943,3346.0
1411,methodological_advance,methodological advance,0.630350,1.700144e-06,2.0,0.0,2.0,0.006060,0.037267,0.442196,469.0,0.442196,2075.0,0.392196,1156.0,0.588450,8268.0
52378,surface_tension_measurement_method,surface tension measurement method,0.254091,1.700144e-06,0.0,1.0,1.0,0.002374,0.045479,0.426123,525.0,0.426123,2244.0,0.376123,1277.0,0.538138,31786.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9324,characterization_method,characterization method,0.735593,1.133429e-06,0.0,1.0,1.0,0.002233,0.044404,0.007478,57344.0,0.007478,57863.0,-0.042522,55219.0,0.154532,56342.0
3972,calculation_method,calculation method,0.622901,6.292427e-01,1.0,2.0,3.0,0.017194,0.066838,0.007411,57358.0,0.022232,56096.0,-0.127768,57049.0,0.075995,57492.0
39481,finite_difference_method,finite difference method,0.692398,1.700144e-06,0.0,1.0,1.0,0.003471,0.053845,0.005593,57634.0,0.005593,57996.0,-0.044407,55372.0,0.135162,56601.0
36016,specific_methodology,specific methodology,0.581624,5.667147e-07,1.0,0.0,1.0,0.002737,0.048251,0.004552,57807.0,0.004552,58064.0,-0.045448,55460.0,0.086315,57331.0


In [49]:
entities_df[entities_df['name'].str.contains('food')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
24499,hygiene_culture_of_agri_food_worker,hygiene culture of agri food worker,0.309986,0.000002,1.0,0.0,1.0,0.001305,0.037327,0.420643,553.0,0.420643,2322.0,0.370643,1330.0,0.590803,7389.0
3573,food_authentication,food authentication,0.328690,0.307819,2.0,1.0,3.0,0.017069,0.066361,0.408346,608.0,0.816693,444.0,0.716693,273.0,0.600425,4413.0
32301,agriculture_and_food_processing_waste_valoriza...,agriculture and food processing waste valoriza...,0.230748,0.000002,1.0,0.0,1.0,0.003058,0.050700,0.336412,1027.0,0.336412,3679.0,0.286412,2253.0,0.609049,2584.0
14415,traditional_food,traditional food,0.568222,0.000002,1.0,0.0,1.0,0.001557,0.039249,0.294323,1479.0,0.294323,4767.0,0.244323,3013.0,0.604388,3463.0
8625,heavy_metal_contamination_in_food_product,heavy metal contamination in food product,0.263353,0.000002,0.0,2.0,2.0,0.009416,0.053417,0.261028,2009.0,0.261028,5991.0,0.211028,3878.0,0.619041,1313.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18101,food_production,food production,0.740107,0.458492,0.0,2.0,2.0,0.009978,0.056119,0.026050,53927.0,0.052099,47074.0,-0.047901,55612.0,0.382921,51740.0
9083,food_production_cost,food production cost,0.650170,0.000001,0.0,2.0,2.0,0.006895,0.041284,0.025803,53978.0,0.025803,55523.0,-0.024197,52808.0,0.522689,36983.0
50964,food_quality,food quality,0.601402,0.458849,0.0,4.0,4.0,0.019783,0.062632,0.019735,55199.0,0.059204,43625.0,-0.090796,56612.0,0.245855,54936.0
46560,food_intake,food intake,0.548316,0.000002,0.0,1.0,1.0,0.002748,0.048333,0.013380,56352.0,0.013380,57309.0,-0.036620,54622.0,0.215620,55428.0


In [50]:
entities_df[entities_df['name'].str.contains('food waste')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
23607,agri_food_wastewater_treatment,agri food wastewater treatment,0.340299,0.000002,0.0,1.0,1.0,0.001541,0.039127,0.075073,30389.0,0.075073,35677.0,0.025073,32367.0,0.613199,1971.0
37982,drying_of_food_waste,drying of food waste,0.442557,0.000002,4.0,0.0,4.0,0.014848,0.046423,0.073874,31043.0,0.073874,36232.0,0.023874,32936.0,0.565255,19336.0
24791,global_food_waste,global food waste,0.593232,0.000003,0.0,1.0,1.0,0.001543,0.039145,0.067919,34513.0,0.067919,39120.0,0.017919,35948.0,0.490366,43969.0
15705,no_food_waste_practice,no food waste practice,0.572441,0.000003,3.0,1.0,4.0,0.015103,0.047262,0.062148,38062.0,0.062148,42068.0,0.012148,39001.0,0.528644,35136.0
37992,food_waste,food waste,0.649087,0.454112,0.0,2.0,2.0,0.008016,0.046679,0.051736,44184.0,0.103472,24669.0,0.003472,43482.0,0.467451,46869.0


In [51]:
entities_df[entities_df['name'].str.contains('virtual reality')].sort_values(by='avg_Score', ascending=False)

,id,name,relevancy_scores,generality_scores,out_degree,in_degree,degree,pagerank,specificity_weighted_pagerank,avg_Score,avg_Rank,sum_Score,sum_Rank,sum_clipped_Score,sum_clipped_Rank,Sigmoid_Score,Sigmoid_Rank
56054,virtual_reality_immersion,virtual reality immersion,0.351619,1.983502e-06,2.0,1.0,3.0,0.013752,0.053714,0.188613,4608.0,0.188613,10389.0,0.138613,7492.0,0.564927,19512.0
11857,virtual_reality_based_exergame,virtual reality based exergame,0.245176,5.667147e-07,3.0,0.0,3.0,0.018393,0.071409,0.175871,5482.0,0.175871,11662.0,0.125871,8583.0,0.552596,25637.0
19174,virtual_reality_immersive_soundscape,virtual reality immersive soundscape,0.297063,1.700144e-06,0.0,1.0,1.0,0.000731,0.032951,0.169603,6036.0,0.169603,12389.0,0.119603,9222.0,0.580804,11533.0
45976,virtual_reality_factor,virtual reality factor,0.391254,1.983502e-06,0.0,1.0,1.0,0.001940,0.042173,0.166155,6331.0,0.166155,12760.0,0.116155,9551.0,0.567108,18433.0
11862,virtual_reality_application,virtual reality application,0.460230,5.595172e-01,7.0,1.0,8.0,0.035923,0.078062,0.160104,6913.0,0.640416,870.0,0.440416,884.0,0.544274,29308.0
12479,virtual_reality_exposure_therapy,virtual reality exposure therapy,0.265130,2.932397e-01,1.0,2.0,3.0,0.016959,0.065940,0.149336,8101.0,0.298673,4653.0,0.198673,4285.0,0.555514,24283.0
4548,realistic_virtual_reality_experience,realistic virtual reality experience,0.465699,1.700144e-06,0.0,1.0,1.0,0.001248,0.036894,0.136847,9870.0,0.136847,17008.0,0.086847,13393.0,0.587742,8564.0
31807,head_mounted_virtual_reality_device,head mounted virtual reality device,0.398984,1.700144e-06,1.0,0.0,1.0,0.002532,0.046683,0.121863,12675.0,0.121863,19950.0,0.071863,16220.0,0.517461,38411.0
7579,social_virtual_reality_platform,social virtual reality platform,0.292720,3.074615e-01,1.0,1.0,2.0,0.007844,0.045854,0.119787,13134.0,0.239573,6975.0,0.139573,7432.0,0.589570,7852.0
15386,virtual_reality_technology,virtual reality technology,0.453475,4.208401e-01,4.0,0.0,4.0,0.018683,0.059018,0.117730,13627.0,0.353189,3316.0,0.203189,4127.0,0.483163,44991.0


In [5]:
import torch
import numpy as np
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import plotly.express as px
from sklearn.cluster import KMeans , DBSCAN
import umap
from sknetwork.clustering import Leiden, Louvain
from sknetwork.data import from_edge_list
from bertopic import BERTopic

D = torch.load(
    '/home/fellah-a/Documents/Text2KG/experiments/topic_modeling/topic_modeling/results/predictions_baseline.pt',
    weights_only=False
)

In [53]:
from src.helpers.torch_utils import create_similarity_graph

In [185]:
edge_index, edge_weight = create_similarity_graph(D['h'] / D['h'].norm(p=2, dim=-1, keepdim=True), th=0.9, batch_size=1024)

In [197]:
def create_similarity_graph( 
    x : Tensor, 
    th : float,
    batch_size : int = 1024
) -> Tuple[Tensor, Tensor]:

    edge_index = []
    edge_weight = []

    for i, batch_i in enumerate(x.split(batch_size)):

        for j, batch_j in enumerate(x.split(batch_size)):

            if i < j:
                continue

            S = batch_i @ batch_j.T

            row, col = torch.argwhere(S >= th).t()

            weights = S[row, col]
            row = row + i * batch_size
            col = col + j * batch_size

            edge_index.append(torch.stack([row, col], dim=0))
            edge_weight.append(weights)

    edge_index = torch.cat(edge_index, dim=-1)
    edge_weight = torch.cat(edge_weight, dim=-1)

    return edge_index, edge_weight

In [209]:
def leiden_cluster(
    x : Tensor,
    th : float = 0.9,
    batch_size : int = 1024,
    weighted : bool = False,
    resolution : int = 1,
    min_cluster_size : int = 10
) -> Tensor:

    ### Create Similarity Graph
    x = x / x.norm(p=2, dim=-1, keepdim=True)
    edge_index, edge_weight = create_similarity_graph(x, th=th, batch_size=batch_size)

    row, col = edge_index
    edge_index = edge_index[:,row != col]
    edge_weight = edge_weight[row != col]

    ### Create data
    if weighted:

        edge_list = list(zip(
            edge_index[0].tolist(),
            edge_index[1].tolist(),
            edge_weight.tolist(),
        ))

    else:

        edge_list = list(zip(
            edge_index[0].tolist(),
            edge_index[1].tolist(),
        ))

    adj = from_edge_list(edge_list, weighted=weighted, shape=(x.size(0), x.size(0)))

    ### Cluster
    communities_probas = torch.from_numpy(Louvain(resolution=resolution).fit_predict_proba(adj))


    ### Filter clusters
    communities = communities_probas.argmax(dim=-1)

    communities, counts = torch.unique(communities, return_counts=True)
    communities_probas = communities_probas[:,communities[counts >= min_cluster_size]]
    communities = communities_probas.argmax(dim=-1)

    ### Compute centers
    centers = scatter(
        src=x,
        index=communities,
        dim=0,
        dim_size=communities_probas.size(1),
        reduce='mean'
    )

    return communities_probas, centers


In [22]:
graph = Graph.load('/home/fellah-a/Documents/Text2KG/data/features/industry_of_future_intfloat--e5-base-v2')

In [23]:
articles_df, entities_df, triples_df = get_dataframes(graph)

In [24]:
topic_model = BERTopic()

topics, probs = topic_model.fit_transform(
    documents=articles_df['abstract'].tolist(),
    embeddings=graph.pygeo['Article'].x.numpy()
)

In [43]:
df

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2004,-1_the_of_and_to,"[the, of, and, to, in, for, is, this, on, with]","[By shifting towards renewable energy sources,..."
1,0,149,0_urban_sound_spaces_heritage,"[urban, sound, spaces, heritage, ambiances, ex...",[The end of World War II finds European urban ...
2,1,123,1_dyes_fluorescence_photophysical_emission,"[dyes, fluorescence, photophysical, emission, ...",[We investigate the emission properties of fou...
3,2,111,2_microalgae_biomass_culture_was,"[microalgae, biomass, culture, was, extraction...",[Microalgae biomass is a promising resource ad...
4,3,99,3_language_models_text_task,"[language, models, text, task, we, ner, nlp, d...",[The recent emergence of Large Language Models...
...,...,...,...,...,...
175,174,11,174_environmental_composite_recycling_thermost...,"[environmental, composite, recycling, thermost...",[The use of composite materials instead of met...
176,175,11,175_ftw_removal_lagoons_ftws,"[ftw, removal, lagoons, ftws, tss, biotricklin...",[Floating Treatment Wetlands (FTW) are a promi...
177,176,11,176_business_alignment_bita_microservices,"[business, alignment, bita, microservices, bus...",[Business-IT Alignment (BITA) is an important ...
178,177,10,177_nfrs_mdd_engineering_conflicts,"[nfrs, mdd, engineering, conflicts, agile, doc...","[In this paper, we report on our 5-year's prac..."


In [42]:
df = topic_model.get_topic_info()
df[df['Topic'] == 50]['Representation'].item()

['chart',
 'charts',
 'ewma',
 'schemes',
 'monitoring',
 'scheme',
 'onesided',
 'run',
 'rl',
 'control']

In [32]:
print('\n'.join(articles_df.iloc[np.array(topics) == 50]['title'].tolist()[:10]))

Online process monitoring under quality data scarcity: Self-starting truncated EWMA schemes for time between events
Two new types of tolerance intervals for sample variances
One-sided truncated EWMA control charts for monitoring two-parameter exponential distributed data
A comprehensive review of time between events control charts: models, applications, and future directions
Enhanced Gamma truncated EWMA-type control charts: One-sided schemes for deterministic and non-deterministic mean shifts in high-quality process monitoring
Exponentially Weighted Moving Average Hypergeometric np Control Scheme
Rank‐Based EWMA TBEA Control Chart
Closed-form expressions of the run-length distribution of the nonparametric double sampling precedence monitoring scheme
The performance of the one-sided truncated exponentially weighted moving average X control chart in the presence of measurement errors
A new multivariate extended homogeneously weighted moving average monitoring scheme incorporated with a 

In [26]:
np.unique(topics, return_counts=True)

(array([ -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,
         12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,
         25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,
         38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,
         51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,
         64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,
         77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
         90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102,
        103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115,
        116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128,
        129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141,
        142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154,
        155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
        168, 169, 170, 171, 172, 173, 174, 175, 176

In [339]:
topics, probs = topic_model.fit_transform(
    documents=articles_df['abstract'].tolist(),
    embeddings=D['h'].numpy()
)

In [123]:
articles_df.iloc[(communities_probas.argmax(dim=-1) == 3).tolist()]

,title,abstract,date,keywords
id,,,,
5461084,Multi-period replenishment planning with suppl...,We consider a multi-period replenishment plann...,2026-01-01T00:00:00Z,"[order_crossover, multiple_sourcing, stochasti..."
5147876,An approximate dynamic programming approach fo...,This work studies a combinatorial optimization...,2026-01-01T00:00:00Z,"[multi_resource_lot_sizing, inventory_bound, l..."
5409410,Load balancing in the mail sorting process: a ...,This paper investigates a load balancing probl...,2025-12-02T00:00:00Z,"[load_smoothing, optimization, assembly_line, ..."
5335875,Stochastic programming for the design and conf...,Reconfigurable assembly lines are designed to ...,2025-10-29T00:00:00Z,"[stochastic_programming, demand_uncertainty, r..."
5408310,Vector bin-packing with variable items represe...,This paper presents an optimization problem ar...,2025-10-01T00:00:00Z,[vector_bin_packing_mail_organization_heuristi...
5452699,Robust multi-manned mixed-model assembly line ...,The manufacturing industry is facing rapid cha...,2025-08-11T00:00:00Z,"[assembly_line_design_and_balancing, adversari..."
5189801,An approximate dynamic programming approach fo...,Lot-Sizing is a class of combinatorial optimiz...,2025-07-28T00:00:00Z,"[dynamic_programming, lot_sizing, multi_stage_..."
5081748,Predictive model-based multi-objective optimiz...,Owing to the realization of advanced manufactu...,2025-06-01T00:00:00Z,"[buffer_allocation, resource_allocation, multi..."
5508602,"Combined Planning of Production Quantities, Re...",We propose a mathematical model for planning t...,2025-01-01T00:00:00Z,"[two_stage_stochastic_programming, mixed_integ..."


In [73]:
D['h'].shape

torch.Size([7783, 768])

In [74]:
torch.unique(edge_index)

tensor([   0,    1,    2,  ..., 1027, 1028, 1029])

In [70]:
D['h']

tensor([[-0.9592,  0.0417,  0.0971,  ..., -1.0539, -1.0181, -0.2664],
        [-1.2512,  2.0739, -0.2882,  ..., -0.4945, -0.9781, -0.0063],
        [-0.8404,  1.0025, -0.5155,  ..., -2.1787, -0.2482, -0.6760],
        ...,
        [-1.5409,  0.8492,  0.7999,  ..., -0.8525,  0.9164,  0.1153],
        [-0.6306,  0.2867, -0.2790,  ..., -0.0786,  0.9995,  0.3925],
        [ 0.5668, -2.0098,  0.4158,  ..., -1.2206,  1.1880, -1.3442]])

In [69]:
communities_probas.shape

torch.Size([1030, 127])

In [ ]:
edge_index

torch.Size([2, 15495250])

In [56]:
edge_index, edge_weight

(tensor([[  0,   0,   0,  ..., 621, 621, 621],
         [  0,   1,   2,  ..., 613, 618, 621]]),
 tensor([ 658.4272,   79.1014,  103.2285,  ...,    1.5253,  195.4283,
         1082.1439]))

In [55]:
raise Exception

Exception: 

In [288]:
pipe = Pipeline([
    ("StandardScaler", StandardScaler()),
    ("TSNE", TSNE())
])

cluster  = Pipeline([
    ("StandardScaler", StandardScaler()),
    ("TSNE", umap.UMAP(n_components=128)),
])

In [342]:
X_projected = cluster.fit_transform(graph.pygeo['Article'].x)
H_projected = cluster.fit_transform(D['h'])

In [354]:
c1 = KMeans(n_clusters=256).fit_predict(X_projected)
c2 = KMeans(n_clusters=256).fit_predict(H_projected)

In [355]:
np.unique(c1, return_counts=True)

(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
        143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
        156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
        169, 170, 171, 172, 173, 174, 175, 176, 177

In [356]:
np.unique(c2, return_counts=True)

(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
        143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
        156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
        169, 170, 171, 172, 173, 174, 175, 176, 177

In [400]:
articles_df.iloc[(c1 == 12).tolist()]['title'].tolist()

['Online process monitoring under quality data scarcity: Self-starting truncated EWMA schemes for time between events',
 'Two new types of tolerance intervals for sample variances',
 'One-sided truncated EWMA control charts for monitoring two-parameter exponential distributed data',
 'A comprehensive review of time between events control charts: models, applications, and future directions',
 'Enhanced Gamma truncated EWMA-type control charts: One-sided schemes for deterministic and non-deterministic mean shifts in high-quality process monitoring',
 'Exponentially Weighted Moving Average Hypergeometric np Control Scheme',
 'Rank‐Based EWMA TBEA Control Chart',
 'Closed-form expressions of the run-length distribution of the nonparametric double sampling precedence monitoring scheme',
 'The performance of the one-sided truncated exponentially weighted moving average X control chart in the presence of measurement errors',
 'A new multivariate extended homogeneously weighted moving average 

In [ ]:
# Topic : 36

In [415]:
articles_df.iloc[(c2 == 36).tolist()]['title'].tolist()

['Revisiting a Classical Real-Time Scheduler for Energy Harvesting Computing Devices',
 'Deadline-Driven Harvesting-aware Scheduling for Autonomous Cyber-Physical Systems',
 'MCED-H: An efficient scheduling policy for energy harvesting mixed-criticality real-time systems',
 'Evaluation of a Power Management Strategy for Autonomous Cyber-Physical Systems with Real-Time Constraints',
 'A Novel Real-Time Mission Scheduler for Energy Neutral Autonomous Mobile Robots',
 'Energy-aware Resource reservation for Multicore architectures',
 'Adaptive High-Performance Optimization Tool for Real-Time Operation of Renewable-Based Virtual Power Plants',
 'ILP representation for Limited Preemption in Energy-Neutral Single-Core Systems',
 'Advanced Research in Real-Time Scheduling for Energy Autonomous Sensor Nodes',
 'Advances and Challenges in Real-time Energy Harvesting Cyber Physical Systems',
 'FP-H: A Real-Time Energy Aware Scheduler with Fixed Priority Assignment for Sustainable Wireless Devices

In [ ]:
c1 = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_projected)
c.max() + 1

np.int64(117)

In [326]:
np.unique(c, return_counts=True)

(array([ -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,
         12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,
         25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,
         38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,
         51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,
         64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,
         77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,
         90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102,
        103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115,
        116]),
 array([1036,   34,  407,  277,  663,   76,  340,   39,  227,  217,   77,
          78,  115,  105,  176,  199,  105,  186,  382,   46,   13,  121,
         164,   31,  285,  381,   36,  134,  142,   17,   87,  117,  148,
          18,   42,    5,   15,   30,    8,  125,   12,   41,   21,   30,
          54,   48,    5,    8, 

In [320]:
X2d = pipe.fit_transform(graph.pygeo['Article'].x)

In [321]:
fig = px.scatter(
    x=X2d[:,0],
    y=X2d[:,1],
    color=c,
    width=750,
    height=750
)

fig.show()

In [ ]:
fig = px.scatter(
    x=X2d[:,0],
    y=X2d[:,1],
    color=c,
    width=750,
    height=750
)

fig.show()

In [1]:
import torch

In [3]:
a = torch.randn(15)
a

tensor([ 0.7868,  2.4171,  0.6265,  0.6781,  0.5388, -0.4169,  0.6026,  0.2080,
        -0.7874, -0.5199, -0.0264,  0.6963,  0.4350,  1.1273,  0.4122])

In [7]:
a.max()

tensor(2.4171)

In [14]:
temp = 0.25
b = a / temp
b = b.softmax(dim=-1)
torch.abs(torch.sum(b * a) - a.max())

tensor(0.0186)